# Quick Start Tutorial

This tutorial explains how to use the higher-level API to perform nucleosome positioning Monte-Carlo simulations using the `nucmc` package.

## Pre-requisite data files

To begin, we download some example methylation footprinting data for the *Oct4* and *XXX* promoter region, with a coverage of ~5000 bp. These data are available from the GitLab repository, and they would have been automatically downloaded when doing `git clone` of the project and stored in the directory `example/experiment/raw_data`. There are four files in this directory:
* `segments.size` - a file containing the size of each DNA fiber segment.
* `chromatin.tsv` - the methylation footprinting scores outputted from ModKit for chromatinized DNA.
* `unmethylated.tsv` - footprinting scores for control samples where the DNA fibers are unmethylated.
* `methylated.tsv` - footprinting scores for control samples where the fibers are fully methylated.

`segments.size` simply contains the identifier and length (in bp) of the DNA segment:

In [2]:
!cat example/experiment/raw_data/segments.size

EcoRI-Oct4-SpeI	5000
SpeI-GREB1-ZraI	5443
SpeI-SFXN2-SacII	5387
SpeI-TFF1-ZraI	5746


In [3]:
!head -n5 example/experiment/raw_data/chromatin.tsv

m84140_251219_135204_s2/189339333/ccs	0	EcoRI-Oct4-SpeI	0	a
m84140_251219_135204_s2/189339333/ccs	1	EcoRI-Oct4-SpeI	0	a
m84140_251219_135204_s2/189339333/ccs	2	EcoRI-Oct4-SpeI	0	a
m84140_251219_135204_s2/189339333/ccs	3	EcoRI-Oct4-SpeI	0	a
m84140_251219_135204_s2/189339333/ccs	6	EcoRI-Oct4-SpeI	0	a


**Note:** It is important to keep the header when outputting the data from ModKit. `nucmc` uses this to identify the relevant data columns to keep for any downstream analysis. In particular, it requires data from the following columns (as per modkit `extract full` output, see <https://nanoporetech.github.io/modkit/intro_extract.html>:

| Column name         | Description                                                 |
|---------------------|-------------------------------------------------------------|
| read_id             | name of the read                                            |
| ref_position        | aligned 0-based reference sequence position                 |
| chrom               | name of aligned contig                                      |
| mod_qual            | probability of the base modifcation in the next column      |
| mod_code            | base modification code from the MM tag                      |
| flag                | FLAG from alignment record                                  |


## Loading the data files and performing normalization

The first step towards performing the simulations is loading the data above into an `MethyPrintExperiment` object. We set up some variables for convenience:

In [1]:
from pathlib import Path

nucbp = 147 # The amount of DNA (in bp) wrapped around a nucleosome

exp_dir = Path("example/experiment")
raw_dir = exp_dir/"raw_data" # Location of the raw methylation scoring data from ModKit
chromsize = raw_dir/"segments.size"
test_file = raw_dir/"chromatin.tsv"
unmeth_file = raw_dir/"unmethylated.tsv"
meth_file = raw_dir/"methylated.tsv"

out_file = exp_dir/"analysis.h5"

The data files above can be loaded using the `nucmc.preprocess` method. Note that you do need to write out the parameter names when passing the arguments.

In [2]:
import nucmc as nc

exp_data = nc.preprocess(binsize = nucbp, 
                         chromsize = chromsize, 
                         test_file = test_file,
                         out_file = out_file,
                         unmeth_file = unmeth_file,
                         meth_file = meth_file,
                         colidx = [0,1,2,3,4])

Reading example/experiment/raw_data/chromatin.tsv ...
Reading example/experiment/raw_data/unmethylated.tsv ...
Reading example/experiment/raw_data/methylated.tsv ...
Normalizing against control samples ...
Normalizing against control samples ...
Normalizing against control samples ...
Normalizing against control samples ...


`exp_data` is a `MethyPrintExperiment` object, which contains only the relevant attributes from the ModKit output that we need for running the simulations. We can use the `raw` property to access these underlying data. For instance, to view the footprinting data for the chromatinized DNA fiber for the *Oct4* segment, we can do the following:

In [3]:
exp_data.raw["EcoRI-Oct4-SpeI"].test_data

,mol_index,pos,mod_qual,mod_code
0,0,0,0.0,a
1,0,1,0.0,a
2,0,2,0.0,a
3,0,3,0.0,a
4,0,6,0.0,a
...,...,...,...,...
2401370,999,4991,0.0,a
2401371,999,4992,0.0,a
2401372,999,4995,0.0,a
2401373,999,4997,0.0,a


As well as the raw data, the object also contains the normalized methylation signal score for each molecule. Since we have provided control datasets (fully methlyated and unmethylated DNA fibers), the normalization was done 

In [13]:
exp_data.analysis["EcoRI-Oct4-SpeI"]["norm"]

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,0.031576,0.031600,0.031690,0.040403,0.040365,0.047568,0.047568,0.047461,0.052890,0.053280,...,-0.004895,-0.004895,-0.004895,-0.004895,-0.004895,-0.004895,-0.004895,-0.004895,-0.004895,-0.004895
1,0.069476,0.069507,0.069628,0.078171,0.078133,0.084687,0.084687,0.084582,0.089230,0.088722,...,0.008923,0.008923,0.008923,0.008923,0.008923,0.008923,0.008923,0.008923,0.008923,0.008923
2,-0.044675,-0.044662,-0.044637,-0.035583,-0.035621,-0.027112,-0.027112,-0.027221,-0.020223,-0.018025,...,0.016099,0.016099,0.016099,0.016099,0.016099,0.016099,0.016099,0.016099,0.016099,0.016099
3,0.107826,0.107863,0.108016,0.116388,0.116351,0.122247,0.122247,0.122144,0.126002,0.124584,...,0.011511,0.011511,0.011511,0.011511,0.011511,0.011511,0.011511,0.011511,0.011511,0.011511
4,-0.007225,-0.007207,-0.007150,0.001737,0.001699,0.009566,0.009566,0.009459,0.015686,0.016996,...,-0.001961,-0.001961,-0.001961,-0.001961,-0.001961,-0.001961,-0.001961,-0.001961,-0.001961,-0.001961
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0.069626,0.069657,0.069778,0.078321,0.078283,0.084834,0.084834,0.084729,0.089374,0.088862,...,0.019836,0.019836,0.019836,0.019836,0.019836,0.019836,0.019836,0.019836,0.019836,0.019836
996,-0.055578,-0.052032,-0.048475,-0.035882,-0.035920,-0.027406,-0.027406,-0.027515,-0.020510,-0.018305,...,-0.017922,-0.017922,-0.017922,-0.017922,-0.017922,-0.017922,-0.017922,-0.017922,-0.017922,-0.017922
997,0.107976,0.108013,0.108167,0.116538,0.116500,0.122394,0.122394,0.122291,0.126146,0.124725,...,0.010919,0.010919,0.010919,0.010919,0.010919,0.010919,0.010919,0.010919,0.010919,0.010919
998,0.145877,0.145919,0.146105,0.154306,0.154269,0.159513,0.159513,0.159411,0.162486,0.160166,...,0.007755,0.007755,0.007755,0.007755,0.007755,0.007755,0.007755,0.007755,0.007755,0.007755


This is now a good checkpoint to save the processed experimental data, which can be done using the `save` method. This stores the relevant raw data and analyses we have done into a single HDF5 file.

In [4]:
exp_h5 = exp_dir/"analysis.h5"
exp_data.save(exp_h5)

We can reload this file later using the `load` method:

In [12]:
from nucmc.experiment.methydata import MethyPrintExperiment
new_exp_h5 = MethyPrintExperiment.load(exp_h5)

## Performing nucleosome positioning sampling simulations
To start the simulations, we need to first create a configuration file specifying the parameter values. At the moment, the package only supports annealing simulations, where the temperature of the system is gradually lowered during the simulation. The list of parameters required are as follows:

| Name          | Description
| ------------- | -----------------------------------------------------------------------------
| nucbp         | The amount of DNA (in bp) occupied by a bound nucleosome (e.g., 147 bp)
| llink         | Length (in bp) of the linker DNA
| mu            | Chemical potential (in units of $k_BT$; see Theory). The energy associated with non-specific binding of nucleosome to DNA
| start_temp    | Start temperature
| end_temp      | End temperature
| ninc_temp     | Number of increments/decrements between start and end temperatures
| nsweep        | Number of "sweeps" or timesteps in the simulation
| print_freq    | Frequency (in sweeps) at which to record simulation data

The file can be in `.json` or `.yml` format. For example, in `.yml` format the file looks as follows:

In [9]:
!cat example/simulation/sim_params.yml

nucbp  = 147
llink  = 50
mu     = 0.0


Alternatively, we can also specify these parameters directly within python by utilizing the `SimSettings` object:

In [10]:
from nucmc.simulation.engine import SimSettings

settings = SimSettings(
    nucbp = 147,
    llink = 50,
    mu = 0.0,
    start_temp = 1.0,
    end_temp = 0.01,
    ninc_temp = 100,
    nsweep = 1000000,
    print_freq = 1000)

To run the simulation, we use the `nucmc.run` method. This

In [ ]:
dataset = nc.run()

Keeping the verbose option `True` will allow a progress bar to be displayed in the terminal when the simulations are running, with an estimation of the time remaining before all jobs are complete. With the parameters specified and manager set up, we can now start the simulations using the `run` method:

## Analyzing the simulation results


In [ ]:
nc.compute(time=tend, dataset)